# Multinomial Naive Bayes for Classification

_**Predict whether a text message is `spam` or `ham` from word-count features in the classic SMS spam classification problem.**_

Develop a program computing prior and conditional (likelihood) probabilities directly from data using Maximum Likelihood Estimation (MLE), classify a test instance by hand-coded application of Bayes' theorem, and repeat the clasification using `sklearn.naive_bayes.MultinomialNB` to verify the predictions match. Also, report accuracy, precision and recall on a held-out test set.

In [1]:
# Imports required packages

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report


The program uses the classic **SMS Spam Collection** dataset (from UCI Machine Learning Repository) that containss a set of SMS messages labelled as `spam` or `ham`. Each message is converted into a bag-of-words vector of word counts, and `MultinomialNB` models $P(x_i \mid c)$ as a multinomial distribution over word counts. Naive Bayes classifies an instance $x = (x_1, \dots, x_n)$ into the class $c$ that maximizes the posterior probability:

$$
P(c \mid x) \propto P(c) \prod_{i=1}^{n} P(x_i \mid c)
$$


## Loading & Analysing the Dataset

In [2]:
# Loads the SMS Spam Collection dataset
sms_data = pd.read_csv("./../../data/sms_spam_collection.csv", sep="\t", header=None, names=["label", "message"])

print("Dataset shape:", sms_data.shape)
print(sms_data["label"].value_counts())
sms_data.head()

Dataset shape: (5572, 2)
label
ham     4825
spam     747
Name: count, dtype: int64


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
# Splits into train/test sets, then converts messages into bag-of-words count vectors

X_sms_train_text, X_sms_test_text, y_sms_train, y_sms_test = train_test_split(
    sms_data["message"], sms_data["label"], test_size=0.25, random_state=42, stratify=sms_data["label"]
)

vectorizer = CountVectorizer(stop_words="english", max_features=3000)
X_sms_train = vectorizer.fit_transform(X_sms_train_text)
X_sms_test = vectorizer.transform(X_sms_test_text)

y_sms_train = y_sms_train.to_numpy()
y_sms_test = y_sms_test.to_numpy()

print("Train set shape:", X_sms_train.shape)
print("Test set shape :", X_sms_test.shape)

Train set shape: (4179, 3000)
Test set shape : (1393, 3000)


## Computing Priors and Word Likelihoods Using MLE (with Laplace Smoothing)

The class priors are estimated as the fraction of training documents in each class. The per-word likelihood for class $c$ is estimated as the relative frequency of word $w$ among all words seen in documents of class $c$:

$$
P(w \mid c) = \frac{\text{count}(w, c) + \alpha}{\sum_{w'} \text{count}(w', c) + \alpha \cdot |V|}
$$

where $|V|$ is the vocabulary size and $\alpha = 1$ is a Laplace (add-one) smoothing constant needed to avoid zero probabilities for words unseen in a given class during training.

In [4]:
# Computes P(class) priors and per-word log-likelihoods using MLE with Laplace smoothing

sms_classes = np.unique(y_sms_train)
vocab_size = X_sms_train.shape[1]
alpha = 1.0

sms_log_priors = {}
sms_feature_log_prob = {}

for c in sms_classes:
    X_c = X_sms_train[y_sms_train == c]
    sms_log_priors[c] = np.log(X_c.shape[0] / X_sms_train.shape[0])

    word_counts = np.asarray(X_c.sum(axis=0)).flatten()
    total_words = word_counts.sum()
    sms_feature_log_prob[c] = np.log((word_counts + alpha) / (total_words + alpha * vocab_size))

for c in sms_classes:
    print(f"Class '{c}': log-prior = {sms_log_priors[c]:.4f}")

Class 'ham': log-prior = -0.1439
Class 'spam': log-prior = -2.0099


## Hand-Coded Classification Using Bayes' Theorem

For a message represented as a word-count vector $x$, the log-posterior score for class $c$ is:

$$
\log P(c \mid x) \;\propto\; \log P(c) + \sum_{w} x_w \cdot \log P(w \mid c)
$$

The predicted class is the one with the highest score. This is computed here as a vectorized dot product between the count vectors and the per-class log-likelihoods for efficiency.

In [5]:
# Hand-coded Multinomial Naive Bayes classifier built directly from the MLE estimates above

def classify_multinomial_nb_manual(X_counts):
    scores = np.zeros((X_counts.shape[0], len(sms_classes)))
    for j, c in enumerate(sms_classes):
        scores[:, j] = sms_log_priors[c] + X_counts.dot(sms_feature_log_prob[c])
    return sms_classes[np.argmax(scores, axis=1)]


# Classifies a single, hand-picked test message to illustrate the calculation
sample_message = X_sms_test_text.iloc[0]
sample_message_label = y_sms_test[0]
sample_message_vector = X_sms_test[0]
sample_message_prediction = classify_multinomial_nb_manual(sample_message_vector)[0]

print("Message  :", sample_message)
print("True label     :", sample_message_label)
print("Predicted label:", sample_message_prediction)

Message  : Blank is Blank. But wat is blank? Lol
True label     : ham
Predicted label: ham


In [6]:
# Fits MultinomialNB on the same training data and predicts on the held-out test set

multi_nb = MultinomialNB(alpha=alpha)
multi_nb.fit(X_sms_train, y_sms_train)
y_sms_pred_sklearn = multi_nb.predict(X_sms_test)

# Classifies the entire test set using the hand-coded classifier
y_sms_pred_manual = classify_multinomial_nb_manual(X_sms_test)

# Verifies that both approaches agree on every test instance
agreement = np.mean(y_sms_pred_manual == y_sms_pred_sklearn)
print(f"Agreement between hand-coded and MultinomialNB predictions: {agreement:.2%}")
assert np.array_equal(y_sms_pred_manual, y_sms_pred_sklearn), "Manual and MultinomialNB predictions differ!"
print("The hand-coded classifier's predictions match MultinomialNB exactly.")

Agreement between hand-coded and MultinomialNB predictions: 100.00%
The hand-coded classifier's predictions match MultinomialNB exactly.


In [7]:
# Reports accuracy, precision and recall on the held-out test set

print(f"Accuracy: {accuracy_score(y_sms_test, y_sms_pred_sklearn):.4f}\n")
print(classification_report(y_sms_test, y_sms_pred_sklearn))

Accuracy: 0.9849

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99      1206
        spam       0.96      0.93      0.94       187

    accuracy                           0.98      1393
   macro avg       0.97      0.96      0.97      1393
weighted avg       0.98      0.98      0.98      1393



## Observations

- Spam classification was proved to be an intuitive, real-world illustration of `MultinomialNB` on discrete count features.

- Priors and class-conditional likelihoods were estimated purely from data via MLE (multinomial word probabilities with Laplace smoothing for SMS spam).

- A from-scratch, hand-coded application of Bayes' theorem reproduced `MultinomialNB`'s predictions exactly on the held-out test sets in both parts.
